### pandas_grouped_stats.py

In [22]:
import argparse
import json
from pathlib import Path

import pandas as pd

MISSING = ["", "na", "n/a", "null", "none", "nan", "missing", "-"]


def numeric_version(series):
    s = series.astype("string").str.strip()
    s = s.str.replace(r"[\$,()%]", "", regex=True).str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")


def infer_type(series):
    non_missing = series.dropna().astype("string").str.strip()
    non_missing = non_missing[~non_missing.str.lower().isin(MISSING)]
    if len(non_missing) == 0:
        return "empty"
    numeric = numeric_version(non_missing)
    if numeric.notna().mean() >= 0.90:
        return "numeric"
    dates = pd.to_datetime(non_missing, errors="coerce")
    if dates.notna().mean() >= 0.70:
        return "date"
    return "categorical"


def numeric_stats(series):
    nums = numeric_version(series).dropna()
    if len(nums) == 0:
        return {"count": 0, "mean": None, "min": None, "max": None, "std": None, "median": None}
    return {
        "count": int(nums.count()),
        "mean": round(float(nums.mean()), 6),
        "min": round(float(nums.min()), 6),
        "max": round(float(nums.max()), 6),
        "std": None if len(nums) < 2 else round(float(nums.std(ddof=1)), 6),
        "median": round(float(nums.median()), 6),
    }


def categorical_stats(series):
    s = series.astype("string").str.strip()
    s = s[~s.isna()]
    s = s[~s.str.lower().isin(MISSING)]
    vc = s.value_counts(dropna=True)
    if len(vc) == 0:
        return {"count": 0, "unique": 0, "mode": None, "mode_freq": 0, "top_5": []}
    return {
        "count": int(s.count()),
        "unique": int(s.nunique()),
        "mode": str(vc.index[0]),
        "mode_freq": int(vc.iloc[0]),
        "top_5": [(str(idx), int(val)) for idx, val in vc.head(5).items()],
    }


def auto_group_columns(df, inferred, max_groups=25):
    choices = []
    for col, typ in inferred.items():
        non_missing = df[col].dropna().astype("string").str.strip()
        unique = int(non_missing.nunique())
        if typ in {"categorical", "date"} and 2 <= unique <= max_groups:
            choices.append((unique, col))
    return [col for _, col in sorted(choices)[:3]]


def grouped_stats(df, group_cols, numeric_cols, cat_cols):
    output = {}
    for group_col in group_cols:
        output[group_col] = {}
        temp = df.copy()
        temp[group_col] = temp[group_col].fillna("(missing)").astype(str)

        for key, group in temp.groupby(group_col, dropna=False):
            output[group_col][str(key)] = {
                "row_count": int(len(group)),
                "numeric": {c: numeric_stats(group[c]) for c in numeric_cols},
                "categorical": {c: categorical_stats(group[c]) for c in cat_cols[:5] if c != group_col},
            }
    return output


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--file", default="2024_fb_ads_president_scored_anon.csv")
    parser.add_argument("--group-cols", default="", help="Comma-separated grouping columns. If omitted, script auto-selects.")
    parser.add_argument("--output", default="pandas_results.json")
    args, unknown = parser.parse_known_args()

    df = pd.read_csv(args.file, dtype="string", keep_default_na=False, na_values=MISSING)

    inferred = {c: infer_type(df[c]) for c in df.columns}
    numeric_cols = [c for c, t in inferred.items() if t == "numeric"]
    cat_cols = [c for c, t in inferred.items() if t in {"categorical", "date"}]

    group_cols = [c.strip() for c in args.group_cols.split(",") if c.strip()]
    if not group_cols:
        group_cols = auto_group_columns(df, inferred)

    result = {
        "script": "pandas_grouped_stats.py",
        "dataset": args.file,
        "overall": {
            "total_rows": int(df.shape[0]),
            "total_columns": int(df.shape[1]),
            "columns": list(df.columns),
            "shape": list(df.shape),
            "dtypes": {c: str(t) for c, t in df.dtypes.items()},
        },
        "missing_values": {
            c: {
                "missing_count": int(df[c].isna().sum()),
                "missing_percent": round(float(df[c].isna().mean() * 100), 4),
            } for c in df.columns
        },
        "inferred_types": inferred,
        "numeric_summary": {c: numeric_stats(df[c]) for c in numeric_cols},
        "categorical_summary": {c: categorical_stats(df[c]) for c in cat_cols},
        "pandas_describe_numeric": df[[c for c in df.columns if c in numeric_cols]].apply(numeric_version).describe().round(6).to_dict() if numeric_cols else {},
        "pandas_describe_all": df.describe(include="all").fillna("").astype(str).to_dict(),
        "grouped_by": group_cols,
        "grouped_summary": grouped_stats(df, group_cols, numeric_cols, cat_cols),
    }

    Path(args.output).write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()


/var/folders/5z/0vb8r1ls42v8ywys6zwf1r0r0000gn/T/ipykernel_57392/1798471438.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(non_missing, errors="coerce")
/var/folders/5z/0vb8r1ls42v8ywys6zwf1r0r0000gn/T/ipykernel_57392/1798471438.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(non_missing, errors="coerce")
/var/folders/5z/0vb8r1ls42v8ywys6zwf1r0r0000gn/T/ipykernel_57392/1798471438.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(non_missing, errors="coerce")
/var/folders/5z/0vb8r1ls42v8ywys6z

{
  "script": "pandas_grouped_stats.py",
  "dataset": "2024_fb_ads_president_scored_anon.csv",
  "overall": {
    "total_rows": 246745,
    "total_columns": 41,
    "columns": [
      "page_id",
      "ad_id",
      "ad_creation_time",
      "bylines",
      "currency",
      "delivery_by_region",
      "demographic_distribution",
      "estimated_audience_size",
      "estimated_impressions",
      "estimated_spend",
      "publisher_platforms",
      "illuminating_scored_message",
      "illuminating_mentions",
      "scam_illuminating",
      "election_integrity_Truth_illuminating",
      "advocacy_msg_type_illuminating",
      "issue_msg_type_illuminating",
      "attack_msg_type_illuminating",
      "image_msg_type_illuminating",
      "cta_msg_type_illuminating",
      "engagement_cta_subtype_illuminating",
      "fundraising_cta_subtype_illuminating",
      "voting_cta_subtype_illuminating",
      "covid_topic_illuminating",
      "economy_topic_illuminating",
      "education_t